<a href="https://colab.research.google.com/github/peach-space/CS3B/blob/main/CS11A_NLP_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis with NLTK

The Natural Language Tool Kit (NLTK) in Python allows us to complete many NLP related tasks. One such task is *sentiment analysis*, which is a prediction of a writer's sentiment, whether positive or negative.

We will apply sentiment analysis to a data set of 2,000 movie reviews. We will predict whether movie reviews are positive or negative using three methods:

- VADER (Valence Aware Dictionary and sEntiment Reasoner): a rule-based sentiment analysis tool that maps words to expected sentiment.
- Naive Bayes classifier
- Logistic regression

Follow along with the assignment instructions and answer the questions listed there.


In [ ]:
# Use this code block to toggle on and off the experiments required for this lab
shuffle_data = False
debalance_classes = False

In [ ]:
import nltk
import random
import numpy as np
import textwrap

# Adjust width as needed for your screen
wrapper = textwrap.TextWrapper(width=70, initial_indent="    ",
                               subsequent_indent="    ", break_long_words=False, break_on_hyphens=False)


from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

In [ ]:
nltk.download('movie_reviews')
nltk.download('vader_lexicon')

In [ ]:
# Convert movie_reviews into a list of raw text documents
# and a list of labels ("pos" or "neg")

documents = []
labels = []

for fileid in movie_reviews.fileids():
    text = movie_reviews.raw(fileid)
    label = movie_reviews.categories(fileid)[0]

    documents.append(text)
    labels.append(label)

print("Number of documents:", len(documents))
print("First label: ", labels[0])
print("First document: ", wrapper.fill(documents[0]))
print()
print("Last label: ", labels[-1])
print("Last document: ", wrapper.fill(documents[-1]))

In [ ]:
# all the negative labels/reviews are first - see it here
print(labels)

In [ ]:
pos_label_count = labels.count('pos')
neg_label_count = labels.count('neg')

print(f"Number of 'pos' reviews: {pos_label_count}")
print(f"Number of 'neg' reviews: {neg_label_count}")

## Experiments

In [ ]:
if debalance_classes:
    # remove 75% of the positive reviews
    pos_label_retained = int(pos_label_count * 0.25)
    labels = labels[0:(neg_label_count + pos_label_retained)]
    documents = documents[0:(neg_label_count + pos_label_retained)]

    new_pos_label_count = labels.count('pos')
    new_neg_label_count = labels.count('neg')

    print(f"Number of 'pos' reviews: {new_pos_label_count}")
    print(f"Number of 'neg' reviews: {new_neg_label_count}")

In [ ]:
if shuffle_data:
    combined = list(zip(documents, labels))
    random.shuffle(combined)
    documents, labels = zip(*combined)

## VADER prediction tool

VADER is a rule-based sentiment analyzer built from a fixed lexicon which applies its predefined dictionary and scoring rules to each document.

In [ ]:
sia = SentimentIntensityAnalyzer()

def vader_predict(text):
    score = sia.polarity_scores(text)["compound"]
    return "pos" if score >= 0 else "neg"

# Generate predictions
vader_preds = [vader_predict(doc) for doc in documents]

# Evaluate
accuracy = accuracy_score(labels, vader_preds)
print("VADER Accuracy:", accuracy)

In [ ]:
# you can look at some example results like this
print(labels[0:5])
print(vader_preds[0:5])
print(labels[-1])
print(vader_preds[-1])

In [ ]:
# summarize results
pos_vader_count = vader_preds.count('pos')
neg_vader_count = vader_preds.count('neg')

print(f"Number of 'pos' predictions: {pos_vader_count}")
print(f"Number of 'neg' predictions: {neg_vader_count}")

In [ ]:
# true vs. predicted labels
cm = confusion_matrix(y_true=labels, y_pred=vader_preds, labels=['pos', 'neg'])
print(cm)

In [ ]:
# another way to see the result above with labels
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["pos", "neg"])
disp.plot()
plt.show()

## TF-IDF

Convert words to tfidf for further use with other algorithms.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.2, random_state=42
)

In [ ]:
len(X_train)

In [ ]:
pos_train_count = y_train.count('pos')
neg_train_count = y_train.count('neg')
pos_test_count = y_test.count('pos')
neg_test_count = y_test.count('neg')

print(f"Number positive training labels: {pos_train_count}")
print(f"Number negative training labels: {neg_train_count}")
print(f"Number positive test labels: {pos_test_count}")
print(f"Number negative test labels: {neg_test_count}")

In [ ]:
# Create a TF-IDF vectorizer and fit it on training data

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF matrix shape:", X_train_tfidf.shape)

## Naive Bayes classification

In [ ]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

nb_preds = nb.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_preds))

In [ ]:
nb_preds_list = nb_preds.tolist()
pos_nb_count = nb_preds_list.count('pos')
neg_nb_count = nb_preds_list.count('neg')

print(f"Number of 'pos' predictions: {pos_nb_count}")
print(f"Number of 'neg' predictions: {neg_nb_count}")

In [ ]:
cm = confusion_matrix(y_true=y_test, y_pred=nb_preds, labels=['pos', 'neg'])
print(cm)

## Logistic Regression classification


In [ ]:
# logistic regression code can go here...

In [ ]:
# Most predictive words - use this once your Logistic Regression is built
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = lr.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-10:]]
top_neg = feature_names[np.argsort(coefs)[:10]]

print("Top Positive Words:", top_pos)
print("Top Negative Words:", top_neg)